In [ ]:
"""
==============================================================================
PROJETO CRÉDITO PF (BRASIL) - ETAPA 1: PREPARAÇÃO, ESTATÍSTICAS E VIF
Módulo responsável pela ingestão de dados, tratamento de quebras de datas do BCB, 
cálculo de estatísticas descritivas (sem Skew/Kurt) e VIF ajustado.
==============================================================================
"""

import logging
from typing import Optional
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings

# ==============================================================================
# CONFIGURAÇÕES GERAIS
# ==============================================================================
warnings.filterwarnings("ignore")

# Configuração de Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger(__name__)

# Configurações Visuais do Matplotlib
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["axes.edgecolor"] = "#4a6175"

# Constantes Globais
MESES_PT = {
    'jan': '01', 'fev': '02', 'mar': '03', 'abr': '04', 'mai': '05', 'jun': '06',
    'jul': '07', 'ago': '08', 'set': '09', 'out': '10', 'nov': '11', 'dez': '12'
}

MODALIDADES_EXPURGO = [
    "Financiamentos à exportação", 
    "Financiamentos à importação",
    "Financiamentos de infraestrutura e desenvolvimento",
    "Financiamentos de títulos e valores mobiliários",
    "Financiamentos com interveniência",
]

# ==============================================================================
# FUNÇÕES AUXILIARES E DE ENGENHARIA DE DADOS
# ==============================================================================
def remover_molduras(ax: Optional[plt.Axes] = None) -> None:
    """Remove as bordas (spines) de um gráfico Matplotlib."""
    if ax is None:
        ax = plt.gca()
    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)

def padronizar_data_bacen(data_str: str) -> str:
    """Converte formatos atípicos de datas do Bacen (ex: 'jul/86') para 'MM/YYYY'."""
    data_limpa = str(data_str).lower().strip()
    for mes_pt, mes_num in MESES_PT.items():
        if data_limpa.startswith(mes_pt):
            partes = data_limpa.split('/')
            if len(partes) > 1:
                ano = partes[1]
                if len(ano) == 2:
                    ano = f"19{ano}" if int(ano) > 50 else f"20{ano}"
                return f"{mes_num}/{ano}"
    return data_limpa

def carregar_csv_local(nome_arquivo: str, nome_coluna: str) -> pd.DataFrame:
    """Busca a série temporal de um arquivo CSV local, padroniza datas e valores."""
    try:
        df = pd.read_csv(f"{nome_arquivo}.csv", sep=";", decimal=",", encoding="latin-1")
    except FileNotFoundError as e:
        logger.error(f"Arquivo '{nome_arquivo}.csv' não foi encontrado.")
        raise e
        
    df.columns = ['data', 'valor']
    df = df[df['data'].astype(str).str.contains(r'/', na=False)].copy()
    
    df['data'] = df['data'].apply(padronizar_data_bacen)
    df['data'] = pd.to_datetime(df['data'], format='%m/%Y', errors='coerce')
    
    if df['valor'].dtype == 'O':
        df['valor'] = df['valor'].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
        
    df[nome_coluna] = pd.to_numeric(df['valor'], errors='coerce')
    df = df.dropna(subset=['data', nome_coluna])
    df['ano_mes'] = df['data'].dt.to_period('M')
    
    df_res = df[['ano_mes', nome_coluna]].drop_duplicates(subset=['ano_mes'])
    logger.info(f"Série '{nome_coluna}' carregada com sucesso ({len(df_res)} registros).")
    return df_res

def mapear_macrobloco_pf_vetorizado(df: pd.DataFrame) -> np.ndarray:
    """Mapeia as submodalidades de crédito para os 4 macroblocos analíticos."""
    sub = df["submodalidade"].astype(str).str.lower()
    mod = df["modalidade"].astype(str).str.lower()

    cond_rotativo = sub.str.contains(
        "crédito rotativo vinculado a cartão|rotativo|cheque especial|sem consignação|"
        "adiantamentos a depositantes|conta garantida|fatura parcelada|saque financiado|"
        "antecipação de fatura", regex=True, na=False
    )
    cond_consignado = sub.str.contains("com consignação|consignado", regex=True, na=False)
    cond_garantia_real = (
        sub.str.contains(
            "habitacional|sfh|veículos automotores|veículos autom.|home equity|"
            "veiculo|veículo|arrendamento financeiro de veículos", regex=True, na=False
        ) | mod.str.contains("imobiliários", regex=True, na=False)
    )

    escolhas = [
        "1. Rotativo / Sem Garantia",
        "2. Consignado / Com Garantia",
        "3. Garantia Real (Imob/Auto)"
    ]
    condicoes = [cond_rotativo, cond_consignado, cond_garantia_real]
    
    return np.select(condicoes, escolhas, default="4. Outros / Renegociados")

# ==============================================================================
# PIPELINE PRINCIPAL
# ==============================================================================
def main():
    logger.info("INICIANDO ETAPA 1: EXTRAÇÃO MACROECONÔMICA E ENGENHARIA DE FEATURES")
    
    # 1. Base Macroeconômica
    df_selic = carregar_csv_local("selic", "selic_media")
    df_juros = carregar_csv_local("juros_pf", "juros_pf")
    df_endiv = carregar_csv_local("endividamento_familias", "endividamento_total")

    df_macro = df_selic.merge(df_juros, on="ano_mes", how="inner")
    df_macro = df_macro.merge(df_endiv, on="ano_mes", how="inner").sort_values("ano_mes").reset_index(drop=True)

    df_macro["spread_bancario"] = df_macro["juros_pf"] - df_macro["selic_media"]
    df_macro["selic_lag6"] = df_macro["selic_media"].shift(6)
    df_macro["endividamento_lag6"] = df_macro["endividamento_total"].shift(6)
    df_macro = df_macro.dropna().reset_index(drop=True)

    # 2. Base de Microdados (Painel SCR)
    logger.info("CARREGANDO E CONSOLIDANDO PAINEL DE CRÉDITO PF")
    cols_load = [
        "data_base", "modalidade", "submodalidade", "carteira_ativa",
        "vencido_de_15_ate_90_dias", "vencido_acima_de_90_dias", "ativo_problematico"
    ]
    
    try:
        df_raw = pd.read_parquet("painel_scr_pf_bruto.parquet", columns=cols_load)
    except FileNotFoundError as e:
        logger.error("Arquivo 'painel_scr_pf_bruto.parquet' não encontrado.")
        raise e

    df_raw = df_raw[~df_raw["modalidade"].isin(MODALIDADES_EXPURGO)]
    df_raw["macrobloco"] = mapear_macrobloco_pf_vetorizado(df_raw)

    df_painel = df_raw.groupby(["data_base", "macrobloco"]).agg(
        carteira_ativa=("carteira_ativa", "sum"),
        vencido_acima_de_90_dias=("vencido_acima_de_90_dias", "sum"),
        ativo_problematico=("ativo_problematico", "sum")
    ).reset_index()

    df_painel["taxa_inadimplencia_pct"] = np.where(
        df_painel["carteira_ativa"] > 0,
        (df_painel["vencido_acima_de_90_dias"] / df_painel["carteira_ativa"]) * 100, 0.0
    )
    df_painel["taxa_ativo_problematico_pct"] = np.where(
        df_painel["carteira_ativa"] > 0,
        (df_painel["ativo_problematico"] / df_painel["carteira_ativa"]) * 100, 0.0
    )

    # 3. Estatísticas Descritivas
    logger.info("GERANDO ESTATÍSTICAS DESCRITIVAS POR MACROBLOCO")
    estatisticas = df_painel.groupby("macrobloco").agg(
        saldo_medio_bi=("carteira_ativa", lambda x: x.mean() / 1e9),
        inad_media=("taxa_inadimplencia_pct", "mean"),
        inad_std=("taxa_inadimplencia_pct", "std"),
        inad_min=("taxa_inadimplencia_pct", "min"),
        inad_mediana=("taxa_inadimplencia_pct", "median"),
        inad_max=("taxa_inadimplencia_pct", "max"),
        ativo_prob_medio=("taxa_ativo_problematico_pct", "mean")
    ).reset_index()

    total_saldo = estatisticas["saldo_medio_bi"].sum()
    estatisticas["part_media_pct"] = (estatisticas["saldo_medio_bi"] / total_saldo) * 100

    estatisticas = estatisticas[[
        "macrobloco", "saldo_medio_bi", "part_media_pct", "inad_media",
        "inad_std", "inad_min", "inad_mediana", "inad_max", "ativo_prob_medio"
    ]].sort_values("saldo_medio_bi", ascending=False)

    print("\n" + "="*80)
    print("ESTATÍSTICAS DESCRITIVAS GLOBAIS:")
    print("="*80)
    print(estatisticas.to_string(index=False))
    print("="*80 + "\n")

    # 4. Diagnóstico de Multicolinearidade e VIF
    logger.info("CALCULANDO MATRIZ DE CORRELAÇÃO E VIF")
    cols_macro_teste = [
        "selic_media", "selic_lag6", "spread_bancario", 
        "endividamento_total", "endividamento_lag6"
    ]

    nomes_vif_plot = {
        "selic_media": "Selic (% a.m.)",
        "selic_lag6": "Selic (Defasagem 6m)",
        "spread_bancario": "Spread Bancário",
        "endividamento_total": "Endiv. Total",
        "endividamento_lag6": "Endiv. (Defasagem 6m)"
    }

    X_vif = df_macro[cols_macro_teste].copy()
    X_vif_const = sm.add_constant(X_vif)

    vif_data = pd.DataFrame()
    vif_data["Variável"] = [nomes_vif_plot.get(c, c) for c in X_vif.columns]
    vif_data["VIF_Centrado"] = [
        variance_inflation_factor(X_vif_const.values, i + 1)
        for i in range(len(cols_macro_teste))
    ]
    
    # Ajuste do VIF para ordem crescente no plot
    vif_data = vif_data.sort_values("VIF_Centrado", ascending=True).reset_index(drop=True)

    # 5. Visualização
    fig = plt.figure(figsize=(18, 7.5), dpi=300)
    gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.2], wspace=0.35) 

    # Matriz de Correlação
    ax1 = fig.add_subplot(gs[0])
    corr_matrix = X_vif.corr()
    corr_matrix.columns = [nomes_vif_plot.get(c, c) for c in corr_matrix.columns]
    corr_matrix.index = [nomes_vif_plot.get(c, c) for c in corr_matrix.index]

    sns.heatmap(
        corr_matrix, annot=True, fmt=".2f", cmap="Blues", square=True, 
        cbar=False, linewidths=0.5, linecolor="white", ax=ax1,
        annot_kws={"size": 10, "weight": "bold"}
    )
    ax1.set_title("1. Matriz de Correlação de Pearson", fontweight="bold", pad=15, fontsize=11)
    ax1.tick_params(axis='x', rotation=0, labelsize=7.5)
    ax1.tick_params(axis='y', rotation=0, labelsize=10)

    # Gráfico de Barras VIF
    ax2 = fig.add_subplot(gs[1])
    cores_vif = ["#d62728" if v >= 10 else "#ff7f0e" if v >= 5 else "#1f77b4" for v in vif_data["VIF_Centrado"]]

    bars = ax2.barh(vif_data["Variável"], vif_data["VIF_Centrado"], color=cores_vif, height=0.6)
    ax2.axvline(5, color="#ff7f0e", linestyle="--", linewidth=1.2, label="Limiar Moderado (VIF = 5.0)")
    ax2.axvline(10, color="#d62728", linestyle=":", linewidth=1.5, label="Limiar Crítico (VIF = 10.0)")

    for bar in bars:
        width = bar.get_width()
        ax2.text(width + 0.3, bar.get_y() + bar.get_height()/2, f"{width:.2f}", 
                 va="center", ha="left", fontweight="bold", fontsize=9)

    ax2.set_title("2. Fator de Inflação de Variância (VIF Centrado Real)", fontweight="bold", pad=15, fontsize=11)
    ax2.set_xlabel("Valor do VIF", fontweight="bold")
    ax2.legend(loc="lower right", frameon=False, fontsize=9) 
    remover_molduras(ax2)

    plt.tight_layout()
    output_filename = "painel_diagnostico_vif_macro_pf.png"
    plt.savefig(output_filename, dpi=300, bbox_inches="tight")
    plt.close()
    
    logger.info(f"Painel salvo com sucesso: '{output_filename}'")
    logger.info("ETAPA 1 CONCLUÍDA.")

if __name__ == "__main__":
    main()